In [39]:
from policyengine_uk import Microsimulation

baseline_enhanced_frs = Microsimulation(
    dataset="hf://policyengine/policyengine-uk-data/enhanced_frs_2023_24.h5"
)

baseline_frs = Microsimulation(
    dataset="hf://policyengine/policyengine-uk-data/frs_2023_24.h5"
)

In [40]:
import pandas as pd

# Official HBAI statistics for 2023/24
# Source: https://www.gov.uk/government/statistics/households-below-average-income-for-financial-years-ending-1995-to-2024
# Data tables: https://assets.publishing.service.gov.uk/media/6836dae29411f0341f323689/HBAI_2324_ods_table_pack.zip

# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.3a (percentages)
# This table shows percentage of individuals below 60% of median income (relative poverty)
# and below 60% of 2010/11 median income adjusted for inflation (absolute poverty)
official_rates_2024 = {
    'Absolute poverty BHC': 15.00,  # Row 36, Column "Absolute low income BHC"
    'Absolute poverty AHC': 18.26,  # Row 36, Column "Absolute low income AHC"
    'Relative poverty BHC': 17.23,  # Row 36, Column "Relative low income BHC"
    'Relative poverty AHC': 21.10,  # Row 36, Column "Relative low income AHC"
}

# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.3b (millions of individuals)
# This table shows number of individuals in poverty (same thresholds as Table 1.3a)
official_populations_2024 = {
    'Absolute poverty BHC': 10.13,  # Row 36, Column "Absolute low income BHC"
    'Absolute poverty AHC': 12.32,  # Row 36, Column "Absolute low income AHC"
    'Relative poverty BHC': 11.63,  # Row 36, Column "Relative low income BHC"
    'Relative poverty AHC': 14.25,  # Row 36, Column "Relative low income AHC"
}

vars_to_load = [
    'person_id',
    'person_weight',
    'household_weight',
    'household_id',
    # Poverty indicators
    'in_poverty_bhc',  # Absolute poverty BHC
    'in_poverty_ahc',  # Absolute poverty AHC
    'in_relative_poverty_bhc',  # Relative poverty BHC
    'in_relative_poverty_ahc',  # Relative poverty AHC
]

# Calculate statistics
def calculate_poverty_stats(df, metric):
    pop = df['person_weight'].sum() / 1e6
    num_in_poverty = df[df[metric] == True]['person_weight'].sum() / 1e6
    rate = (num_in_poverty / pop) * 100
    return rate, num_in_poverty, pop

# Calculate for Enhanced FRS
print("Loading Enhanced FRS data for 2024...")
enhanced_dict = {}
for var in vars_to_load:
    enhanced_dict[var] = baseline_enhanced_frs.calculate(var, 2024, map_to="person").values
enhanced_df = pd.DataFrame(enhanced_dict)

# Calculate for standard FRS
print("Loading standard FRS data for 2024...")
frs_dict = {}
for var in vars_to_load:
    frs_dict[var] = baseline_frs.calculate(var, 2024, map_to="person").values
frs_df = pd.DataFrame(frs_dict)

# Population totals
_, _, frs_pop = calculate_poverty_stats(frs_df, 'in_poverty_bhc')
_, _, enhanced_pop = calculate_poverty_stats(enhanced_df, 'in_poverty_bhc')

# Build comparison tables
rate_data = []
pop_data = []

metrics_mapping = [
    ('in_relative_poverty_bhc', 'Relative poverty BHC'),
    ('in_relative_poverty_ahc', 'Relative poverty AHC'),
    ('in_poverty_bhc', 'Absolute poverty BHC'),
    ('in_poverty_ahc', 'Absolute poverty AHC'),
]

for metric, label in metrics_mapping:
    frs_rate, frs_num, _ = calculate_poverty_stats(frs_df, metric)
    enhanced_rate, enhanced_num, _ = calculate_poverty_stats(enhanced_df, metric)
    official_rate = official_rates_2024.get(label, None)
    official_pop = official_populations_2024.get(label, None)
    
    # Rate table
    rate_row = {
        'Measure': label,
        'Official (%)': f"{official_rate:.2f}" if official_rate else "N/A",
        'Standard FRS (%)': f"{frs_rate:.2f}",
        'Enhanced FRS (%)': f"{enhanced_rate:.2f}",
        'Standard FRS - Official (pp)': f"{frs_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Official (pp)': f"{enhanced_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Standard FRS (pp)': f"{enhanced_rate - frs_rate:+.2f}",
    }
    rate_data.append(rate_row)
    
    # Population table
    pop_row = {
        'Measure': label,
        'Official (M)': f"{official_pop:.2f}" if official_pop else "N/A",
        'Standard FRS (M)': f"{frs_num:.2f}",
        'Enhanced FRS (M)': f"{enhanced_num:.2f}",
        'Standard FRS - Official (M)': f"{frs_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Official (M)': f"{enhanced_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Standard FRS (M)': f"{enhanced_num - frs_num:+.2f}",
    }
    pop_data.append(pop_row)

rate_table = pd.DataFrame(rate_data)
pop_table = pd.DataFrame(pop_data)

print("\n" + "=" * 140)
print("POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)")
print("=" * 140)
print(f"\nTotal Population:")
print(f"  Official (HBAI):  67.51 million")
print(f"  Standard FRS:     {frs_pop:.2f} million")
print(f"  Enhanced FRS:     {enhanced_pop:.2f} million")
print("\n")
print(rate_table.to_string(index=False))
print("\n" + "=" * 140)

print("\n" + "=" * 140)
print("POVERTY POPULATIONS COMPARISON: 2024 (FY 2023/24) (millions)")
print("=" * 140)
print("\n")
print(pop_table.to_string(index=False))
print("\n" + "=" * 140)

print("\nKEY FINDINGS:")
print("-" * 140)
print("• Standard FRS - Official: Shows how accurately standard FRS replicates official poverty statistics")
print("  → Target: Within ±1.0 percentage point")
print("• Enhanced FRS - Official: Shows how much Enhanced FRS diverges from official statistics")
print("• Enhanced FRS - Standard FRS: Shows the impact of data enhancements (reweighting, imputation, etc.)")
print("\nSOURCES:")
print("• Relative/Absolute poverty: HBAI Tables 1.3a (rates) and 1.3b (populations)")
print("=" * 140)

Loading Enhanced FRS data for 2024...
Loading standard FRS data for 2024...

POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)

Total Population:
  Official (HBAI):  67.51 million
  Standard FRS:     68.23 million
  Enhanced FRS:     68.01 million


             Measure Official (%) Standard FRS (%) Enhanced FRS (%) Standard FRS - Official (pp) Enhanced FRS - Official (pp) Enhanced FRS - Standard FRS (pp)
Relative poverty BHC        17.23            18.39            16.95                        +1.16                        -0.28                            -1.44
Relative poverty AHC        21.10            23.18            21.13                        +2.08                        +0.03                            -2.05
Absolute poverty BHC        15.00            16.13            12.26                        +1.13                        -2.74                            -3.87
Absolute poverty AHC        18.26            19.47            17.16                        +1.21                    

In [42]:
# WORKING-AGE ADULTS POVERTY ANALYSIS

# Official HBAI working-age adults poverty statistics for 2023/24
# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.5a (percentages)
official_working_age_rates_2024 = {
    'Relative poverty BHC': 14.69,
    'Absolute poverty BHC': 12.94,
    'Relative poverty AHC': 19.35,
    'Absolute poverty AHC': 16.85,
}

# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.5b (millions of working-age adults)
official_working_age_populations_2024 = {
    'Relative poverty BHC': 6.00,
    'Absolute poverty BHC': 5.29,
    'Relative poverty AHC': 7.91,
    'Absolute poverty AHC': 6.89,
}

# Calculate working-age adults poverty statistics
def calculate_working_age_poverty_stats(df, metric):
    # Filter for working-age adults (16-64 years)
    working_age_df = df[(df['age'] >= 16) & (df['age'] <= 64)]
    pop = working_age_df['person_weight'].sum() / 1e6
    num_in_poverty = working_age_df[working_age_df[metric] == True]['person_weight'].sum() / 1e6
    rate = (num_in_poverty / pop) * 100
    return rate, num_in_poverty, pop

# Load age variable for both datasets
print("Loading age variable...")
enhanced_df['age'] = baseline_enhanced_frs.calculate('age', 2024, map_to="person").values
frs_df['age'] = baseline_frs.calculate('age', 2024, map_to="person").values

# Build working-age adults poverty comparison tables
working_age_rate_data = []
working_age_pop_data = []

for metric, label in metrics_mapping:
    frs_rate, frs_num, _ = calculate_working_age_poverty_stats(frs_df, metric)
    enhanced_rate, enhanced_num, _ = calculate_working_age_poverty_stats(enhanced_df, metric)
    official_rate = official_working_age_rates_2024.get(label, None)
    official_pop = official_working_age_populations_2024.get(label, None)
    
    # Rate table
    rate_row = {
        'Measure': label,
        'Official (%)': f"{official_rate:.2f}" if official_rate else "N/A",
        'Standard FRS (%)': f"{frs_rate:.2f}",
        'Enhanced FRS (%)': f"{enhanced_rate:.2f}",
        'Standard FRS - Official (pp)': f"{frs_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Official (pp)': f"{enhanced_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Standard FRS (pp)': f"{enhanced_rate - frs_rate:+.2f}",
    }
    working_age_rate_data.append(rate_row)
    
    # Population table
    pop_row = {
        'Measure': label,
        'Official (M)': f"{official_pop:.2f}" if official_pop else "N/A",
        'Standard FRS (M)': f"{frs_num:.2f}",
        'Enhanced FRS (M)': f"{enhanced_num:.2f}",
        'Standard FRS - Official (M)': f"{frs_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Official (M)': f"{enhanced_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Standard FRS (M)': f"{enhanced_num - frs_num:+.2f}",
    }
    working_age_pop_data.append(pop_row)

working_age_rate_table = pd.DataFrame(working_age_rate_data)
working_age_pop_table = pd.DataFrame(working_age_pop_data)

# Get working-age population totals
_, _, frs_working_age_pop = calculate_working_age_poverty_stats(frs_df, 'in_poverty_bhc')
_, _, enhanced_working_age_pop = calculate_working_age_poverty_stats(enhanced_df, 'in_poverty_bhc')

print("\n" + "=" * 140)
print("WORKING-AGE ADULTS POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)")
print("=" * 140)
print(f"\nTotal Working-Age Adults Population (ages 16-64):")
print(f"  Official (HBAI):  40.87 million")
print(f"  Standard FRS:     {frs_working_age_pop:.2f} million")
print(f"  Enhanced FRS:     {enhanced_working_age_pop:.2f} million")
print("\n")
print(working_age_rate_table.to_string(index=False))
print("\n" + "=" * 140)

print("\n" + "=" * 140)
print("WORKING-AGE ADULTS POVERTY POPULATIONS COMPARISON: 2024 (FY 2023/24) (millions)")
print("=" * 140)
print("\n")
print(working_age_pop_table.to_string(index=False))
print("\n" + "=" * 140)

print("\nKEY FINDINGS:")
print("-" * 140)
print("• Working-age adults poverty rates are lower than children poverty rates but higher than pensioner poverty rates")
print("• Standard FRS - Official: Shows how accurately standard FRS replicates official working-age poverty statistics")
print("  → Target: Within ±1.0 percentage point")
print("• Enhanced FRS - Official: Shows how much Enhanced FRS diverges from official statistics")
print("• Enhanced FRS - Standard FRS: Shows the impact of data enhancements on working-age poverty estimates")
print("\nSOURCES:")
print("• Working-age adults poverty statistics: HBAI Tables 1.5a (rates) and 1.5b (populations)")
print("=" * 140)

Loading age variable...

WORKING-AGE ADULTS POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)

Total Working-Age Adults Population (ages 16-64):
  Official (HBAI):  40.87 million
  Standard FRS:     42.29 million
  Enhanced FRS:     43.62 million


             Measure Official (%) Standard FRS (%) Enhanced FRS (%) Standard FRS - Official (pp) Enhanced FRS - Official (pp) Enhanced FRS - Standard FRS (pp)
Relative poverty BHC        14.69            15.83            16.28                        +1.14                        +1.59                            +0.45
Relative poverty AHC        19.35            21.40            21.26                        +2.05                        +1.91                            -0.14
Absolute poverty BHC        12.94            13.93            11.54                        +0.99                        -1.40                            -2.39
Absolute poverty AHC        16.85            18.59            18.01                        +1.74                     

In [41]:
# CHILDREN POVERTY ANALYSIS

# Official HBAI children poverty statistics for 2023/24
# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.4a (percentages)
official_children_rates_2024 = {
    'Relative poverty BHC': 23.19,
    'Absolute poverty BHC': 19.99,
    'Relative poverty AHC': 30.52,
    'Absolute poverty AHC': 26.39,
}

# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.4b (millions of children)
official_children_populations_2024 = {
    'Relative poverty BHC': 3.38,
    'Absolute poverty BHC': 2.92,
    'Relative poverty AHC': 4.45,
    'Absolute poverty AHC': 3.85,
}

# Calculate children poverty statistics
def calculate_children_poverty_stats(df, metric):
    # Filter for children only
    children_df = df[df['is_child'] == True]
    pop = children_df['person_weight'].sum() / 1e6
    num_in_poverty = children_df[children_df[metric] == True]['person_weight'].sum() / 1e6
    rate = (num_in_poverty / pop) * 100
    return rate, num_in_poverty, pop

# Load is_child variable for both datasets
print("Loading is_child variable...")
enhanced_df['is_child'] = baseline_enhanced_frs.calculate('is_child', 2024, map_to="person").values
frs_df['is_child'] = baseline_frs.calculate('is_child', 2024, map_to="person").values

# Build children poverty comparison tables
children_rate_data = []
children_pop_data = []

for metric, label in metrics_mapping:
    frs_rate, frs_num, _ = calculate_children_poverty_stats(frs_df, metric)
    enhanced_rate, enhanced_num, _ = calculate_children_poverty_stats(enhanced_df, metric)
    official_rate = official_children_rates_2024.get(label, None)
    official_pop = official_children_populations_2024.get(label, None)
    
    # Rate table
    rate_row = {
        'Measure': label,
        'Official (%)': f"{official_rate:.2f}" if official_rate else "N/A",
        'Standard FRS (%)': f"{frs_rate:.2f}",
        'Enhanced FRS (%)': f"{enhanced_rate:.2f}",
        'Standard FRS - Official (pp)': f"{frs_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Official (pp)': f"{enhanced_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Standard FRS (pp)': f"{enhanced_rate - frs_rate:+.2f}",
    }
    children_rate_data.append(rate_row)
    
    # Population table
    pop_row = {
        'Measure': label,
        'Official (M)': f"{official_pop:.2f}" if official_pop else "N/A",
        'Standard FRS (M)': f"{frs_num:.2f}",
        'Enhanced FRS (M)': f"{enhanced_num:.2f}",
        'Standard FRS - Official (M)': f"{frs_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Official (M)': f"{enhanced_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Standard FRS (M)': f"{enhanced_num - frs_num:+.2f}",
    }
    children_pop_data.append(pop_row)

children_rate_table = pd.DataFrame(children_rate_data)
children_pop_table = pd.DataFrame(children_pop_data)

# Get children population totals
_, _, frs_children_pop = calculate_children_poverty_stats(frs_df, 'in_poverty_bhc')
_, _, enhanced_children_pop = calculate_children_poverty_stats(enhanced_df, 'in_poverty_bhc')

print("\n" + "=" * 140)
print("CHILDREN POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)")
print("=" * 140)
print(f"\nTotal Children Population:")
print(f"  Official (HBAI):  14.59 million")
print(f"  Standard FRS:     {frs_children_pop:.2f} million")
print(f"  Enhanced FRS:     {enhanced_children_pop:.2f} million")
print("\n")
print(children_rate_table.to_string(index=False))
print("\n" + "=" * 140)

print("\n" + "=" * 140)
print("CHILDREN POVERTY POPULATIONS COMPARISON: 2024 (FY 2023/24) (millions)")
print("=" * 140)
print("\n")
print(children_pop_table.to_string(index=False))
print("\n" + "=" * 140)

print("\nKEY FINDINGS:")
print("-" * 140)
print("• Children poverty rates are consistently higher than overall population poverty rates")
print("• Standard FRS - Official: Shows how accurately standard FRS replicates official children poverty statistics")
print("  → Target: Within ±1.0 percentage point")
print("• Enhanced FRS - Official: Shows how much Enhanced FRS diverges from official statistics")
print("• Enhanced FRS - Standard FRS: Shows the impact of data enhancements on children poverty estimates")
print("\nSOURCES:")
print("• Children poverty statistics: HBAI Tables 1.4a (rates) and 1.4b (populations)")
print("=" * 140)

Loading is_child variable...

CHILDREN POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)

Total Children Population:
  Official (HBAI):  14.59 million
  Standard FRS:     14.58 million
  Enhanced FRS:     14.31 million


             Measure Official (%) Standard FRS (%) Enhanced FRS (%) Standard FRS - Official (pp) Enhanced FRS - Official (pp) Enhanced FRS - Standard FRS (pp)
Relative poverty BHC        23.19            25.17            23.40                        +1.98                        +0.21                            -1.77
Relative poverty AHC        30.52            34.03            30.36                        +3.51                        -0.16                            -3.67
Absolute poverty BHC        19.99            22.08            17.58                        +2.09                        -2.41                            -4.50
Absolute poverty AHC        26.39            29.11            24.44                        +2.72                        -1.95                    

In [46]:
# PENSIONERS POVERTY ANALYSIS

# Official HBAI pensioners poverty statistics for 2023/24
# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.6a (percentages)
official_pensioners_rates_2024 = {
    'Relative poverty BHC': 18.64,
    'Absolute poverty BHC': 15.96,
    'Relative poverty AHC': 15.66,
    'Absolute poverty AHC': 13.16,
}

# From summary-hbai-1994-95-2023-24-tables.xlsx, Table 1.6b (millions of pensioners)
official_pensioners_populations_2024 = {
    'Relative poverty BHC': 2.25,
    'Absolute poverty BHC': 1.92,
    'Relative poverty AHC': 1.89,
    'Absolute poverty AHC': 1.59,
}

# Calculate pensioners poverty statistics
def calculate_pensioners_poverty_stats(df, metric):
    # Filter for pensioners (State Pension Age)
    pensioners_df = df[df['is_SP_age'] == True]
    pop = pensioners_df['person_weight'].sum() / 1e6
    num_in_poverty = pensioners_df[pensioners_df[metric] == True]['person_weight'].sum() / 1e6
    rate = (num_in_poverty / pop) * 100
    return rate, num_in_poverty, pop

# Load is_SP_age variable for both datasets
print("Loading is_SP_age variable...")
enhanced_df['is_SP_age'] = baseline_enhanced_frs.calculate('is_SP_age', 2024, map_to="person").values
frs_df['is_SP_age'] = baseline_frs.calculate('is_SP_age', 2024, map_to="person").values

# Build pensioners poverty comparison tables
pensioners_rate_data = []
pensioners_pop_data = []

for metric, label in metrics_mapping:
    frs_rate, frs_num, _ = calculate_pensioners_poverty_stats(frs_df, metric)
    enhanced_rate, enhanced_num, _ = calculate_pensioners_poverty_stats(enhanced_df, metric)
    official_rate = official_pensioners_rates_2024.get(label, None)
    official_pop = official_pensioners_populations_2024.get(label, None)
    
    # Rate table
    rate_row = {
        'Measure': label,
        'Official (%)': f"{official_rate:.2f}" if official_rate else "N/A",
        'Standard FRS (%)': f"{frs_rate:.2f}",
        'Enhanced FRS (%)': f"{enhanced_rate:.2f}",
        'Standard FRS - Official (pp)': f"{frs_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Official (pp)': f"{enhanced_rate - official_rate:+.2f}" if official_rate else "N/A",
        'Enhanced FRS - Standard FRS (pp)': f"{enhanced_rate - frs_rate:+.2f}",
    }
    pensioners_rate_data.append(rate_row)
    
    # Population table
    pop_row = {
        'Measure': label,
        'Official (M)': f"{official_pop:.2f}" if official_pop else "N/A",
        'Standard FRS (M)': f"{frs_num:.2f}",
        'Enhanced FRS (M)': f"{enhanced_num:.2f}",
        'Standard FRS - Official (M)': f"{frs_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Official (M)': f"{enhanced_num - official_pop:+.2f}" if official_pop else "N/A",
        'Enhanced FRS - Standard FRS (M)': f"{enhanced_num - frs_num:+.2f}",
    }
    pensioners_pop_data.append(pop_row)

pensioners_rate_table = pd.DataFrame(pensioners_rate_data)
pensioners_pop_table = pd.DataFrame(pensioners_pop_data)

# Get pensioners population totals
_, _, frs_pensioners_pop = calculate_pensioners_poverty_stats(frs_df, 'in_poverty_bhc')
_, _, enhanced_pensioners_pop = calculate_pensioners_poverty_stats(enhanced_df, 'in_poverty_bhc')

print("\n" + "=" * 140)
print("PENSIONERS POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)")
print("=" * 140)
print(f"\nTotal Pensioners Population (State Pension Age):")
print(f"  Official (HBAI):  12.06 million")
print(f"  Standard FRS:     {frs_pensioners_pop:.2f} million")
print(f"  Enhanced FRS:     {enhanced_pensioners_pop:.2f} million")
print("\n")
print(pensioners_rate_table.to_string(index=False))
print("\n" + "=" * 140)

print("\n" + "=" * 140)
print("PENSIONERS POVERTY POPULATIONS COMPARISON: 2024 (FY 2023/24) (millions)")
print("=" * 140)
print("\n")
print(pensioners_pop_table.to_string(index=False))
print("\n" + "=" * 140)

print("\nKEY FINDINGS:")
print("-" * 140)
print("• Pensioners have the lowest poverty rates among demographic groups")
print("• Note: Pensioners show lower poverty AHC than BHC (unlike other groups) due to higher home ownership rates")
print("• Standard FRS - Official: Shows how accurately standard FRS replicates official pensioner poverty statistics")
print("  → Target: Within ±1.0 percentage point")
print("• Enhanced FRS - Official: Shows how much Enhanced FRS diverges from official statistics")
print("• Enhanced FRS - Standard FRS: Shows the impact of data enhancements on pensioner poverty estimates")
print("\nSOURCES:")
print("• Pensioners poverty statistics: HBAI Tables 1.6a (rates) and 1.6b (populations)")
print("=" * 140)

Loading is_SP_age variable...

PENSIONERS POVERTY RATES COMPARISON: 2024 (FY 2023/24) (%)

Total Pensioners Population (State Pension Age):
  Official (HBAI):  12.06 million
  Standard FRS:     12.17 million
  Enhanced FRS:     10.86 million


             Measure Official (%) Standard FRS (%) Enhanced FRS (%) Standard FRS - Official (pp) Enhanced FRS - Official (pp) Enhanced FRS - Standard FRS (pp)
Relative poverty BHC        18.64            19.97            12.71                        +1.33                        -5.93                            -7.26
Relative poverty AHC        15.66            18.09            10.84                        +2.43                        -4.82                            -7.25
Absolute poverty BHC        15.96            17.30             8.96                        +1.34                        -7.00                            -8.33
Absolute poverty AHC        13.16            12.55             6.18                        -0.61                        